In [1]:
import os
import pandas as pd
import cv2
import xml.etree.ElementTree as ET

In [10]:
ROOT_PATH = "dataset/"
CLEAN_PATH = 'clean_dataset/'
IMAGE_CLEAN_PATH = 'clean_dataset/images/'

IMAGE_DIR = os.path.join(ROOT_PATH, 'images')
ANNOT_DIR = os.path.join(ROOT_PATH, 'annotations')

In [3]:
def extract_annotation(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    records = []
    filename = root.find('filename').text
    
    for obj in root.findall('object'):
        record = {
            'filename': filename,
            'class': obj.find('name').text,
            'xmin': int(obj.find('bndbox/xmin').text),
            'ymin': int(obj.find('bndbox/ymin').text),
            'xmax': int(obj.find('bndbox/xmax').text),
            'ymax': int(obj.find('bndbox/ymax').text)
        }
        records.append(record)
    
    return records

In [6]:
all_records = []
for annot_file in os.listdir(ANNOT_DIR):
    if annot_file.endswith('.xml'):
        annot_path = os.path.join(ANNOT_DIR, annot_file)
        records = extract_annotation(annot_path)
        all_records.extend(records)

In [11]:
def crop_image(image_counter, record):
    img_path = os.path.join(IMAGE_DIR, record["filename"])
    image = cv2.imread(img_path)
    
    xmin = record["xmin"]
    ymin = record["ymin"]
    xmax = record["xmax"]
    ymax = record["ymax"]

    pad = 4
    xmin = max(0, xmin - pad)
    ymin = max(0, ymin - pad)
    xmax = min(image.shape[1], xmax + pad)
    ymax = min(image.shape[0], ymax + pad)

    if xmax <= xmin or ymax <= ymin:
        return image_counter
    
    crop = image[ymin:ymax, xmin:xmax]

    if crop.size == 0:
        return image_counter
    
    output_filename = f"image_{image_counter}.png"
    output_path = os.path.join(IMAGE_CLEAN_PATH, output_filename)
    cv2.imwrite(output_path, crop)
    
    return image_counter + 1

In [13]:
image_counter = 0
csv_data = []

for record in all_records:
    csv_data.append({
        'image_index': image_counter,
        'filename': f"image_{image_counter}.png",
        'class': record["class"],
    })
    image_counter = crop_image(image_counter, record)

df_labels = pd.DataFrame(csv_data)
csv_path = os.path.join(CLEAN_PATH, 'annotation.csv')
df_labels.to_csv(csv_path, index=False)

print(df_labels.head())

   image_index     filename           class
0            0  image_0.png     With Helmet
1            1  image_1.png     With Helmet
2            2  image_2.png  Without Helmet
3            3  image_3.png  Without Helmet
4            4  image_4.png  Without Helmet
